In [1]:
import pandas as pd
import openpyxl
import numpy as np
import os


In [2]:
# payroll report file import

# Automatically detect Downloads folder
downloads = os.path.join(os.path.expanduser("~"), "Downloads")

# Build full file paths
report_path = os.path.join(downloads, "fullpaymentdetails.csv")

# Read the CSV files
pay_report = pd.read_csv(report_path)


### Enter Payroll Reconciliation numbers

In [3]:
# enter numbers form Summary payroll report
gross_pay_rec = 41295.61
net_pay_rec = 30769.10
# ensure the df has the same values
print("Summary Report: Gross Pay: ", gross_pay_rec)
print("Payroll File: Gross Salary:", pay_report['Gross salary'].sum().round(2) )
print("Summary Report: Net Pay: ", net_pay_rec)
print("Payroll File: Net Pay:", pay_report['Net pay'].sum().round(2) )

Summary Report: Gross Pay:  41295.61
Payroll File: Gross Salary: 41295.61
Summary Report: Net Pay:  30769.1
Payroll File: Net Pay: 30769.1


In [4]:
# create df for workings
df = pay_report.copy()

#### Clean data


In [5]:
# Replace missing values with 0
df.fillna(0, inplace=True)

In [6]:
# columns that have been previosly used in reports and are included in report calculations
expected_columns=['Employee Reference', 'NI Number', 'Forename', 'Surname', 'Start date',
       'Leaving date', 'Job title', 'Job category', 'Contracted hours',
       'Contracted weeks', 'Full time hours', 'Full time weeks', 'FTE',
       'Paygrade name', 'Payscale name', 'FTE annual salary', 'Annual actual',
       'Gross salary', 'Post Gross salary', 'Projected monthly salary',
       'Adhoc corrections', 'Accrued Leave', 'Acting Up', 'Bonus',
       'Compensation Payment (Tax & NI-free)', 'General Expenses', 'Holiday',
       'Industrial Action', 'KIT Days', 'Overtime', 'Overtime 1.3333',
       'Overtime 1.5', 'Overtime 2.0', 'PILON', 'Recovery',
       'Redundancy over 30k (taxable)', 'Redundancy under 30K (Tax free)',
       'Supply Teaching', 'TLR Recovery', 'Unpaid Leave (hourly)', 'Allowance',
       'Fire Marshall', 'First Aid', 'Honorarium', 'R&R', 'Responsibility',
       'SEN', 'TLR', 'TLR 1', 'TLR 2', 'TLR 3', 'GMB Union', 'Childcare',
       'Emp Advance', 'Bicycle Incentive', 'Cycle to Work', 'Computing Scheme', 'Unison', 'Computing Scheme (Tech Scheme).1',
       'Net Pay Recovery', 'Computing Scheme (Tech Scheme)', 'Salary sacrifice overheads', 'Salary sacrifice',
       'Overpayment Recovery', 'Recovery.1', 'Recovery.2', 'Gross deduction correction',
       'Absence deduction', 'OMP', 'SMP', 'SMP Deduction', 'ShPP',
       'ShPP Deduction', 'UPL Deduction', 'OSP', 'SSP', 'SSP deduction', 'OPP',
       'SPP', 'SPP deduction', 'Tax code', 'Tax paid',
       'Student loan', 'Postgraduate loan', 'Pension Support',
       'ER Pension - Support', 'Pension Teachers', 'ER Pension Teachers',
       'Pension AVC Total', 'NI Code', 'Employee NI Total',
       'Employer NI Total', 'Total statutory recovered and compensated',
       'Total ER cost', 'Net pay', 'Dept code', 'Backpay',
       'Occupational Maternity Pay', 'Statutory Maternity Pay', 'Maternity Leave Deduction', 
       'Shared Parental Leave Deduction', 'Sick Days Deduction', 'SSP Offset',
        'Acting Up 1', 'Acting Up', 'Faster Payment Net Recovery', 'Short Unpaid Leave',
        'Holiday Pay (pensionable, hourly rate)']


<div style="border:1px solid white; border-radius:6px; overflow:hidden;">
  <div style="background:#f69697; color:white; padding:8px 12px; font-weight:bold;">
    IMPORTANT!!! Check for new columns
  </div>
  <div style="background:#f4cccc; padding:12px;">
    Any new columns should be added to calculations in Grouping Columns section (if required) and to columns_in_calculations list
  </div>
</div>


In [7]:
missing_columns=[col for col in df.columns if col not in expected_columns]
if missing_columns:
    #print("New columns: ", missing_columns)
    print(f"\033[46mNew columns: {missing_columns}\033[0m")
else:
    print("\033[43mNo new columns\033[0m")

New columns: ['Honorarium (non-pensionable)']


In [8]:
# Assign nill values for currently missing columns
for col in expected_columns:
    if col not in df.columns:
        print(col)
        df[col] = 0

NI Number
Start date
Leaving date
Job title
Job category
Contracted hours
Contracted weeks
Full time hours
Full time weeks
FTE
Paygrade name
Payscale name
Bicycle Incentive
Computing Scheme (Tech Scheme).1
Recovery.1
OMP
SMP
SMP Deduction
ShPP Deduction
SSP deduction
Tax code


<div style="border:1px solid white; border-radius:6px; overflow:hidden;">
  <div style="background:#f69697; color:white; padding:8px 12px; font-weight:bold;">
    IMPORTANT!!! Adjustments control
  </div>
  <div style="background:#f4cccc; padding:14px; ">
    The adjustment lines must stay commented out during initial processing. Uncomment and apply ONLY after reconciliation identifies valid mismatches.
  </div>
</div>


In [9]:
# Adjustment lines
df.loc[(df['Dept code'] == 4000) & (df['Employee Reference'] == 569881), 'Adhoc corrections'] += 250
df.loc[(df['Dept code'] == 1200) & (df['Employee Reference'] == 569972), 'Projected monthly salary'] = 476.45

# Grouping Columns

In [10]:
# Grouping Employees' details
df['Employee ID'] = df['Employee Reference']
df['Employee Name'] = df['Surname'] + " " + df['Forename']

In [11]:
# Grouping Additions and Deductions
df['Monthly Salary'] = df['Projected monthly salary'] 

df['Adhoc Pay'] = df['Adhoc corrections'] + df['Recovery'] + df['Holiday'] + df['Overpayment Recovery'] \
+ df['Recovery.1'] + df['Gross deduction correction'] + df['PILON'] + df['Compensation Payment (Tax & NI-free)'] + df['Backpay']\
+ df['KIT Days'] + df['Acting Up (DNU)'] + df['Acting Up'] + df['Holiday Pay (pensionable, hourly rate)']

df['Allowance'] = df['Allowance'] 
df['Overtime'] = df['Overtime'] + df['Overtime 1.3333'] + df['Overtime 1.5'] + df['Overtime 2.0']
df['Unpaid leave'] = df['Unpaid Leave (hourly)'] + df['Absence deduction'] + df['UPL Deduction'] + df['Sick Days Deduction'] +df['Auth Unpaid Leave <14 days lgps']

df['SSP&OSP'] = df['SSP'] + df['OSP'] + df['SSP Offset']
df['OS/FL Deductions'] = df['SMP Deduction'] + df['ShPP Deduction'] + df['SPP deduction'] + df['SSP deduction'] + df['Shared Parental Leave Deduction'] + df['Maternity Leave Deduction']
df['OMP'] = df['Occupational Maternity Pay'] + df['OMP'] + df['OPP']
df['SMP'] = df['SMP'] + df['Statutory Maternity Pay'] + df['SPP']

In [12]:
# Grouping statutory deductions
df["EE's NIC"] = df['Employee NI Total']
df["ER's NIC"] = df['Employer NI Total']
df["ER's Recovery"] = df['Total statutory recovered and compensated']
df["EE's Pension"] = df['Pension Support'] + df['Pension Teachers']
df["ER's Pension"] = df['ER Pension - Support'] + df['ER Pension Teachers']
df['Tax'] = df['Tax paid']

In [13]:
# Grouping other deductions (After Tax on the report)
df['Union Sub'] = - df['Unison'] - df['GMB Union']  # positive on the report
df['Loans'] = - df['Student loan'] - df['Postgraduate loan'] # positive on the report

df['HomeTech'] =  df['Computing Scheme (Tech Scheme)'] + df['Bicycle Incentive'] + df['Computing Scheme (Tech Scheme).1'] \
+ df['Salary sacrifice']# negative on the report df['Salary sacrifice overheads'] 

df['Advance Recovery'] = - df['Net Pay Recovery'] - df['Faster Payment Net Recovery']# positive on the report


## GROSS AMOUNT

In [14]:
df['Gross Pay'] = df[['Projected monthly salary',
   'Adhoc Pay', 'Allowance', 'Responsibility', 'Honorarium', 'SEN', 'TLR', 'TLR 2', 'TLR 3', 
   'Overtime', 'Unpaid leave', 'SMP', 'OMP', 'ShPP', 'SSP&OSP', 'OS/FL Deductions']].sum(axis=1)
#print(df['Gross Pay'].sum())

### Apportionment (NIC, TAX)

In the original dataset, NIC and TAX amounts are recorded only against the first role. The code below redistributes these values proportionally across all roles (when a staff memebr has 2 or more roles). By apportioning NIC and TAX, the calculation ensures that costs are allocated accurately between departments. This approach prevents distortions in departmental reporting and provides a fair representation of payroll expenses.

In [15]:
# Step 1: Calculate totals per employee - Salary, EE NIC and ER NIC, Tax
df['Total Gross Pay'] = df.groupby('Employee ID')['Gross Pay'].transform('sum')
df['Total EE NIC'] = df.groupby('Employee ID')["EE's NIC"].transform('sum')
df['Total ER NIC'] = df.groupby('Employee ID')["ER's NIC"].transform('sum')
df['Total Tax'] = df.groupby('Employee ID')["Tax"].transform('sum')

# Step 2: Apportion 
df['App EE NIC'] = np.where(      # NumPy function works like a vectorised if-else
    df['Total Gross Pay'] == 0,   # condition, it check if the Total Gross Pay is zero, to prevent division by zero errors later
    df["EE's NIC"],               # if condition is True
    df['Total EE NIC'] * df['Gross Pay'] / df['Total Gross Pay'] # if condition is False
)

df['App ER NIC'] = np.where(
    df['Total Gross Pay'] == 0,
    df["ER's NIC"], 
    df['Total ER NIC'] * df['Gross Pay'] / df['Total Gross Pay']
)

df['App Tax'] = np.where(
    df['Total Gross Pay'] == 0,
    df['Tax'],
    df['Total Tax'] * df['Gross Pay'] / df['Total Gross Pay']
)

# Step 3: Update the original columns
df["EE's NIC"] = df['App EE NIC'].round(2)
df["ER's NIC"] = df['App ER NIC'].round(2)
df['Tax'] = df['App Tax'].round(2)

# Drop helper columns
df_final = df.drop(columns=['Total Gross Pay', 'Total EE NIC', 'Total ER NIC', 
                            'Total Tax', 'App EE NIC', 'App ER NIC', 'App Tax'])

## NET PAY

In [16]:
df['Net Pay'] = df['Gross Pay'] - df["EE's NIC"] - df["EE's Pension"] - df['Tax'] \
+ df['Union Sub'] + df['Loans'] + df['HomeTech'] + df['Advance Recovery']

#### Net Pay Reconciliation (Summary Report vs Payroll Details)

In [17]:
df['Net Pay Rec'] = df['Net Pay'] - df['Net pay']
df['Net Pay Rec'] = df['Net Pay Rec'].astype('float').round(2)
total_net = df.groupby('Employee Reference')['Net Pay Rec'].sum().reset_index()
total_net[total_net['Net Pay Rec'] !=0]


,Employee Reference,Net Pay Rec


In [18]:
# ======================================================
# Check by Employee's ID (payslip's calculation)
# ======================================================
# columns to include in payslip calculation
cols = ['Employee ID', 'Employee Name', 'Job title', 'FTE annual salary', 'Annual actual', 'Dept code', 'Monthly Salary',
   'Adhoc Pay', 'Allowance', 'Responsibility', 'Honorarium', 'SEN', 'TLR', 'TLR 2', 'TLR 3', 'Overtime', 'Unpaid leave', 'SMP', 'OMP', 'ShPP', 'SSP&OSP', 'OS/FL Deductions', 
    'Gross Pay',
    "EE's NIC", "ER's NIC", "ER's Recovery", "EE's Pension", "ER's Pension", 'Tax',
    'Union Sub', 'Loans', 'HomeTech', 'Advance Recovery',
    'Net Pay'
   ]
existing_cols = [c for c in cols if c in df.columns]
check = df[existing_cols]

# filter by employee's number
result = check[check['Employee ID'] == 569972].iloc[0] # ************** ENTER EMPLOEE'S ID ***********************

# print (result)
skip_cols=['NI Number', 'Start date', 'Job category', 'Contracted hours', 'Contracted weeks', 'Full time hours', 'Full time weeks', 'FTE', 'Paygrade name T-Ups']
for col, value in result.items():
    if value != 0.0 and col not in skip_cols:
        print(col, value)
        

Employee ID 569972
Employee Name Ricci Matteo
FTE annual salary 24324.45
Annual actual 5717.4
Dept code 1200
Monthly Salary 476.45
Gross Pay 476.45
EE's NIC 4.16
ER's NIC 57.94
EE's Pension 26.2
ER's Pension 73.37
Tax 5.18
Net Pay 440.90999999999997


### Manual Verification of Mismatches
All mismatched records identified by the script will now be checked against each individual payslip to confirm the correct values.
Where discrepancies are confirmed, the necessary adjustments will be applied to the payroll reconciliation report using the code section provided below. This ensures that all payroll figures are fully validated and accurately reflected in the final reconciliation output.

### Adding missing values

##### Step 1 Values before adjustments

In [19]:
employee_ids = [569881, 569972]
check_cols = ['Employee Name', 'Employee ID', 'Adhoc Pay', 'Monthly Salary', 'Dept code']
df_filtered = df[df['Employee ID'].isin(employee_ids)]
print(df_filtered[check_cols])

   Employee Name  Employee ID  Adhoc Pay  Monthly Salary  Dept code
1   Qureshi Amir       569881     346.18         3795.73       4000
16  Ricci Matteo       569972       0.00          476.45       1200


##### Step 2 Adjustments

In [20]:
# **************************************************************************
# ******** ADD ADJUSTMENTS AT ROW 13 & RESTART KERNEL **********************
# **************************************************************************
# ********* reconcile here, the final adjustmensts to row[13] **************

# ID 569881	-250.00 => pay correction on payslip => add to Adhoc Pay column 
# df.loc[(df['Dept code'] == 4000) & (df['Employee Reference'] == 569881), 'Adhoc corrections'] += 250

# ID 569972	0.01 => payslip gross amount £476.45, report - £476.46  => overright the amount on the report 
# df.loc[(df['Dept code'] == 1200) & (df['Employee Reference'] == 569972), 'Projected monthly salary'] = 476.45

##### Step 3 Check if the value has been added

In [21]:
df_filtered = df[df['Employee ID'].isin(employee_ids)]
print(df_filtered[check_cols])

   Employee Name  Employee ID  Adhoc Pay  Monthly Salary  Dept code
1   Qureshi Amir       569881     346.18         3795.73       4000
16  Ricci Matteo       569972       0.00          476.45       1200


## REPORT 1 - Full Details


In [22]:
cols2 = ['Employee ID', 'Employee Name', 'Job title', 'FTE annual salary', 'Annual actual', 'Dept code', 'Monthly Salary',
   'Adhoc Pay', 'Allowance', 'Responsibility', 'Honorarium', 'SEN', 'TLR', 'TLR 2', 'TLR 3', 'Overtime', 'Unpaid leave', 'SMP', 'OMP', 'ShPP', 'SSP&OSP', 'OS/FL Deductions', 
    'Gross Pay',
    "EE's NIC", "ER's NIC", "ER's Recovery", "EE's Pension", "ER's Pension", 'Tax',
    'Union Sub', 'Loans', 'HomeTech', 'Advance Recovery',
    'Net Pay'
   ]

existing_cols2 = [c for c in cols2 if c in df.columns]
report = df.loc[:, existing_cols2].copy()

In [23]:
employee_ids = [569881, 569972]
check_cols = ['Employee Name', 'Employee ID', 'Adhoc Pay', 'Monthly Salary', 'Dept code']
df_filtered = report[report['Employee ID'].isin(employee_ids)]
print(df_filtered[check_cols])

   Employee Name  Employee ID  Adhoc Pay  Monthly Salary  Dept code
1   Qureshi Amir       569881     346.18         3795.73       4000
16  Ricci Matteo       569972       0.00          476.45       1200


#### Remove empty rows


In [24]:
cols_to_check = ['Monthly Salary','Adhoc Pay', 'Allowance', 'Responsibility', 'Honorarium', 
                 'SEN', 'TLR', 'TLR 2', 'TLR 3', 'Overtime', 'Unpaid leave', 'SMP', 'OMP', 'ShPP', 'SSP&OSP', 'OS/FL Deductions', 
                 'Gross Pay', "EE's NIC", "ER's NIC", "ER's Recovery", "EE's Pension", "ER's Pension", 'Tax',
                 'Union Sub', 'Loans', 'HomeTech', 'Advance Recovery', 'Net Pay'
]
report= report[~(report[cols_to_check] == 0).all(axis=1)] #exclude rows with no pay values, leave only active roles

In [37]:
# Check Point - Gross Amount
column_sum = report.iloc[:, 6:23].sum() # up till Gross Total
print(column_sum)
print ("Diffrence between Summary Report :", round(report['Gross Pay'].sum()-gross_pay_rec + report['HomeTech'].sum(),2))

Monthly Salary      41379.34
Adhoc Pay             556.54
Allowance               0.00
Responsibility          0.00
Honorarium             19.44
SEN                    74.41
TLR                   118.04
TLR 2                 274.48
TLR 3                 173.02
Overtime              210.41
Unpaid leave          -12.95
SMP                   395.11
OMP                   342.15
ShPP                  259.09
SSP&OSP                 0.00
OS/FL Deductions    -2431.49
Gross Pay           41357.59
dtype: float64
Diffrence between Summary Report : -0.0


In [26]:
# CheckPoint: Statutory Deductions - Match the Summary report (manually)
column_sum = report.iloc[:, 23:29].sum() # NIC, Pen and Tax
print(column_sum) 

EE's NIC         1937.99
ER's NIC         5563.02
ER's Recovery     601.85
EE's Pension     2545.73
ER's Pension     5610.12
Tax              5298.00
dtype: float64


In [27]:
# Check Point: Other deductions and NET Pay - - Match the Summary report (manually)
column_sum = report.iloc[:, 29:35].sum() # Deductions after tax
print(column_sum)

Union Sub              -4.31
Loans                -467.38
HomeTech              -61.98
Advance Recovery     -273.10
Net Pay             30769.10
dtype: float64


## REPORT 2 - Summary by Department

In [30]:
# all after tax deductions in one column
report['After Tax'] = report[['Union Sub', 'Loans', 'HomeTech', 'Advance Recovery']].sum(axis=1)

# column required for the summary report
dep_full = report[['Dept code', 'Gross Pay', "EE's NIC", "ER's NIC", "ER's Recovery", "EE's Pension",
                   "ER's Pension", 'Tax', 'After Tax', 'Net Pay'
]].copy()

# NIC and NIC recovery into one column
dep_full["ER's NIC"] = dep_full["ER's NIC"] - dep_full["ER's Recovery"] 
dep_full_drop = dep_full.drop("ER's Recovery", axis=1, inplace=True)

# Group by Cost Code
dep_summary = dep_full.groupby('Dept code').sum(numeric_only=True).reset_index()
#Add Nominal Codes
dep_summary['NIC N/Code'] = dep_summary['Dept code'].astype(int) + 1 
dep_summary['Pen N/Code'] = dep_summary['Dept code'].astype(int) + 2

# Rearrange Columns
col_to_move = 'NIC N/Code'
cols = list(dep_summary.columns)
cols.remove(col_to_move)
cols.insert(3, col_to_move)
dep_summary = dep_summary[cols]
col_to_move = 'Pen N/Code'
cols.remove(col_to_move)
cols.insert(6, col_to_move)
dep_summary = dep_summary[cols]
dep_summary['Total Cost to School'] = dep_summary['Gross Pay'] + dep_summary["ER's NIC"] + dep_summary["ER's Pension"]

#print report

dep_summary

,Dept code,Gross Pay,EE's NIC,NIC N/Code,ER's NIC,EE's Pension,Pen N/Code,ER's Pension,Tax,After Tax,Net Pay,Total Cost to School
0,1200,2383.62,41.80,1201,275.88,121.44,1202,373.30,133.21,0.00,2087.17,3032.80
1,1700,1820.29,14.66,1701,167.17,100.09,1702,280.31,67.00,0.00,1638.54,2267.77
2,4000,14524.92,726.20,4001,1489.70,794.14,4002,2417.61,2262.91,-720.08,10021.59,18432.23
3,4100,14521.39,602.52,4101,1792.29,1201.63,4102,1750.90,1416.83,-31.85,11268.56,18064.58
4,7000,2637.51,275.04,7001,523.09,169.28,7002,410.93,627.90,0.00,1565.29,3571.53
5,8000,5469.86,277.77,8001,713.04,159.15,8002,377.07,790.15,-54.84,4187.95,6559.97


## REPORTS - Export to Exel

In [31]:
# Calculate column sums (excluding non-numeric if needed)
total_values =[]
for col in report.columns:
    if pd.api.types.is_numeric_dtype(report[col]):
        total_values.append(report[col].sum())
    else:
        total_values.append('Total') 

# Append to original report 
report_with_total = pd.concat([report, pd.DataFrame([total_values], columns=report.columns)], ignore_index=True)

# EXPORT TO EXCEL
# Step 1. Automatically detect Downloads folder
downloads = os.path.join(os.path.expanduser("~"), "Downloads")

# Step 2. Base filename
base_name = 'Payroll_report_summary'
extension = '.xlsx'

# Step 3. Original file name
filename = f"{base_name}{extension}"
full_path = os.path.join(downloads, filename)

# Step 4. Check if file exists and increment if needed
counter = 1
while os.path.exists(full_path):
    filename = f"{base_name}({counter}){extension}"
    full_path = os.path.join(downloads, filename)
    counter += 1
    
# step 5 Export ot Excel
with pd.ExcelWriter(full_path, engine='openpyxl') as writer:
    report_with_total.to_excel(writer, sheet_name="Full Details", index=False)
    dep_summary.to_excel(writer, sheet_name="Cost code", index=False)

print(f"Report saved to {full_path}")

Report saved to C:\Users\anzel\Downloads\Payroll_report_summary(7).xlsx
